<a href="https://colab.research.google.com/github/YukthaGowda/AUTOIMMUNE-DISEASE/blob/main/DEPLOYMENT_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **This Google Colab notebook demonstrates a basic sample model for Autoimmune Disease Prediction, trained using XGBoost, and includes a simple UI for inputting symptoms and viewing predictions.**

In [ ]:
# IMPORTS
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier

from ipywidgets import interact, FloatSlider, Dropdown


In [ ]:
# LOAD DATA
df = pd.read_csv("Final_Balanced_Autoimmune_Disorder_Dataset.csv")

df.head()

,Patient_ID,Age,Gender,Diagnosis,Sickness_Duration_Months,RBC_Count,Hemoglobin,Hematocrit,MCV,MCH,...,Anti_TIF1,Anti_epidermal_basement_membrane_IgA,Anti_OmpC,pANCA,Anti_tissue_transglutaminase,anti_Scl_70,Anti_Mi2,Anti_parietal_cell,Progesterone_antibodies,Anti_Sm
0,6,62,Male,Autoimmune orchitis,41,4.75,13.37,43.11,101.91,28.41,...,0,0,0,0,0,0,0,0,0,0
1,20,54,Female,Autoimmune orchitis,41,4.32,10.76,39.92,95.96,28.22,...,0,0,0,0,0,0,0,0,0,0
2,46,34,Male,Autoimmune orchitis,86,4.42,11.91,38.38,80.56,28.40,...,0,0,0,0,0,0,0,0,0,0
3,108,22,Male,Autoimmune orchitis,43,4.33,12.72,39.99,84.71,26.67,...,0,0,0,0,0,0,0,0,0,0
4,142,20,Female,Autoimmune orchitis,50,3.99,11.07,43.58,89.87,30.64,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
# SELECT FEATURES

num_cols = ['Age', 'CRP', 'ESR', 'Hemoglobin',
            'RBC_Count', 'Hematocrit', 'MCV', 'MCH', 'MCHC']

cat_cols = ['Gender']

target_col = "Diagnosis"   # <-- change if needed

X = df[num_cols + cat_cols].copy()
y_raw = df[target_col]

In [ ]:
# ENCODE TARGET
le = LabelEncoder()
y = le.fit_transform(y_raw)

n_classes = len(le.classes_)
n_classes


6

In [ ]:
# TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


In [ ]:
# BUILD PREPROCESSOR
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

ct = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", ohe, cat_cols),
], remainder="drop")


In [ ]:
# BUILD XGBOOST MODEL

best_model = Pipeline([
    ("prep", ct),
    ("model", XGBClassifier(
        objective="multi:softprob",
        num_class=n_classes,
        n_estimators=300,
        learning_rate=0.1,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=42
    ))
])

best_model.fit(X_train, y_train)

print("XGBoost model trained successfully!")


XGBoost model trained successfully!


In [ ]:
# PREDICTION FUNCTION

def predict_autoimmune(best_model, input_dict):
    X_input = pd.DataFrame([input_dict])

    pred_class = best_model.predict(X_input)[0]
    predicted_label = le.inverse_transform([pred_class])[0]

    probs = best_model.predict_proba(X_input)[0]
    top3_idx = probs.argsort()[-3:][::-1]

    top3 = [(le.inverse_transform([i])[0], float(probs[i])) for i in top3_idx]

    return predicted_label, top3


In [ ]:
from ipywidgets import VBox, HBox, Button, FloatSlider, Dropdown, Output

# Create Slider Widgets
Age = FloatSlider(description="Age", min=18, max=85, step=1, value=40)
CRP = FloatSlider(description="CRP", min=0, max=10, step=0.1, value=1)
ESR = FloatSlider(description="ESR", min=0, max=100, step=0.1, value=20)
Hemoglobin = FloatSlider(description="Hemoglobin", min=5, max=20, step=0.1, value=13)
RBC_Count = FloatSlider(description="RBC_Count", min=3, max=6, step=0.1, value=4.5)
Hematocrit = FloatSlider(description="Hematocrit", min=20, max=60, step=0.1, value=40)
MCV = FloatSlider(description="MCV", min=50, max=120, step=0.1, value=90)
MCH = FloatSlider(description="MCH", min=20, max=40, step=0.1, value=30)
MCHC = FloatSlider(description="MCHC", min=20, max=40, step=0.1, value=33)
Gender = Dropdown(description="Gender", options=["Male", "Female"], value="Male")

# Buttons
predict_btn = Button(description="🔮 Predict", button_style='success')
clear_btn = Button(description="🧹 Clear", button_style='danger')

output = Output()

# Predict button
def on_predict_clicked(b):
    output.clear_output()
    user_input = {
        "Age": Age.value,
        "CRP": CRP.value,
        "ESR": ESR.value,
        "Hemoglobin": Hemoglobin.value,
        "RBC_Count": RBC_Count.value,
        "Hematocrit": Hematocrit.value,
        "MCV": MCV.value,
        "MCH": MCH.value,
        "MCHC": MCHC.value,
        "Gender": Gender.value
    }

    pred_label, top3 = predict_autoimmune(best_model, user_input)

    with output:
        print("🔥 PREDICTION RESULT 🔥")
        print("Most likely autoimmune disease:", pred_label)
        print("\nTop 3 probabilities:")
        for disease, p in top3:
            print(f"  {disease}: {p:.3f}")

predict_btn.on_click(on_predict_clicked)

# Clear button
def on_clear_clicked(b):
    Age.value = 40
    CRP.value = 1
    ESR.value = 20
    Hemoglobin.value = 13
    RBC_Count.value = 4.5
    Hematocrit.value = 40
    MCV.value = 90
    MCH.value = 30
    MCHC.value = 33
    Gender.value = "Male"
    output.clear_output()

clear_btn.on_click(on_clear_clicked)

# DISPLAY UI
ui = VBox([
    Age, CRP, ESR, Hemoglobin, RBC_Count, Hematocrit,
    MCV, MCH, MCHC, Gender,
    HBox([predict_btn, clear_btn]),
    output
])

ui
